# 03 · Sliding Window

> Una finestra contigua `[l, r]` che si allarga a destra e si restringe a sinistra, mantenendo uno stato (conteggi, set, massimo) aggiornato in `O(1)` per passo invece di ricalcolarlo.
> Idee per i singoli problemi: [Pattern algoritmici › Sliding window](../Fundamentals/Pattern%20algoritmici.md#sliding-window)

## Problemi → cosa ti serve

| Problema | Concetti | Sintassi |
|---|---|---|
| Best Time to Buy and Sell Stock | minimo visto finora + miglior differenza | `min_price = min(min_price, p)`, `float("inf")` |
| Longest Substring Without Repeating Characters | finestra variabile: restringi finché c'è un duplicato | `set`, `discard`, oppure `dict` carattere → ultimo indice |
| Longest Repeating Character Replacement | finestra valida se `lunghezza − frequenza max ≤ k` | `dict.get(ch, 0) + 1`, `max(counts.values())` |
| Permutation in String | finestra **fissa** lunga `len(s1)`: confronta i conteggi | `Counter`, `[0] * 26`, `ord(ch) - ord("a")` |
| Minimum Window Substring | finestra variabile: allarga finché valida, poi restringi | `Counter`, contatore `have`/`need`, `s[l : r + 1]` |
| Sliding Window Maximum | deque monotona di indici | `deque`, `append`, `pop`, `popleft`, `dq[0]` |

## Sintassi al volo

| Serve | Sintassi |
|---|---|
| infinito | `float("inf")`, `math.inf`, `-math.inf` |
| conteggio con default | `counts[ch] = counts.get(ch, 0) + 1` |
| decrementa ed elimina lo zero | `counts[ch] -= 1` · `if counts[ch] == 0: del counts[ch]` |
| lunghezza della finestra | `r - l + 1` |
| sottostringa della finestra | `s[l : r + 1]` |
| deque | `from collections import deque` · `append`/`appendleft`, `pop`/`popleft` in `O(1)`, `dq[0]`, `dq[-1]` |

## Minimo corrente (finestra degenere)

Il caso più semplice: per ogni giorno ti basta il minimo dei giorni precedenti, che aggiorni mentre scorri. Puoi vederlo come una finestra in cui `l` è il giorno del minimo.

In [1]:
daily_price = [7, 1, 5, 3, 6, 4]

lowest = float("inf")
best_gain = 0
for price in daily_price:
    lowest = min(lowest, price)
    best_gain = max(best_gain, price - lowest)
print(lowest, best_gain)  # -> 1 5

1 5


## Finestra variabile: lo scheletro

1. Allarga: aggiungi `s[r]` allo stato.
2. Finché la finestra **non** è valida, togli `s[l]` dallo stato e fai `l += 1`.
3. Aggiorna la risposta con la finestra corrente.

Ogni indice entra ed esce una sola volta: `O(n)` anche con il `while` interno.

In [2]:
# Sessione di navigazione più lunga senza pagine ripetute.
pages = ["home", "mug", "cap", "mug", "cart", "pay"]

in_window: set[str] = set()
l = 0
longest = 0
for r, page in enumerate(pages):
    while page in in_window:       # non valida: restringi da sinistra
        in_window.discard(pages[l])
        l += 1
    in_window.add(page)
    longest = max(longest, r - l + 1)
print(longest)  # -> 4  (cap, mug, cart, pay)

4


## Lo stato della finestra: conteggi

Quando la validità dipende dalle frequenze, mantieni un dict di conteggi. `max(counts.values())` costa `O(26)` con un alfabeto fisso: va bene.

In [3]:
window = "AABAB"
counts: dict[str, int] = {}
for ch in window:
    counts[ch] = counts.get(ch, 0) + 1
print(counts, max(counts.values()))  # -> {'A': 3, 'B': 2} 3

# uscita da sinistra
leaving = window[0]
counts[leaving] -= 1
if counts[leaving] == 0:
    del counts[leaving]  # utile se confronti i dict o conti le chiavi distinte
print(counts)  # -> {'A': 2, 'B': 2}

{'A': 3, 'B': 2} 3
{'A': 2, 'B': 2}


## Finestra fissa e confronto di conteggi

Con lunghezza fissa `k`: aggiungi l'elemento che entra (`s[r]`) e togli quello che esce (`s[r - k]`). Due array di 26 si confrontano con `==` in `O(26)`.

In [4]:
target = "abc"
text = "xbcaz"
k = len(target)

need = [0] * 26
for ch in target:
    need[ord(ch) - ord("a")] += 1

window = [0] * 26
for r, ch in enumerate(text):
    window[ord(ch) - ord("a")] += 1
    if r >= k:
        window[ord(text[r - k]) - ord("a")] -= 1  # esce l'elemento di k passi fa
    if r >= k - 1 and window == need:
        print("trovata in", r - k + 1)  # -> trovata in 1

trovata in 1


## `Counter` per "contiene tutti i caratteri richiesti"

Per la finestra minima che copre un insieme di caratteri, conviene non confrontare tutto il dict a ogni passo. Tieni `need` (quanti caratteri distinti devi soddisfare) e `have` (quanti lo sono già): `have` cambia solo quando un conteggio **raggiunge** o **scende sotto** la soglia.

In [5]:
from collections import Counter

required = Counter("AABC")
print(required, len(required))  # -> Counter({'A': 2, 'B': 1, 'C': 1}) 3

window = Counter()
have = 0
for ch in "BANA":
    window[ch] += 1
    if ch in required and window[ch] == required[ch]:  # appena soddisfatto
        have += 1
print(have)  # -> 2: B e A sono soddisfatti, manca C

print(Counter("AAB") - Counter("AB"))  # -> Counter({'A': 1}): la sottrazione elimina i ≤ 0

Counter({'A': 2, 'B': 1, 'C': 1}) 3
2
Counter({'A': 1})


## Deque monotona: massimo della finestra

Una `deque` di **indici** con valori decrescenti dal fronte al fondo:
- prima di inserire `i`, togli dal fondo gli indici con valore `≤ nums[i]` (non saranno mai più il massimo);
- togli dal fronte l'indice uscito dalla finestra;
- il massimo è `nums[dq[0]]`.

Ogni indice entra ed esce una volta: `O(n)`.

In [6]:
from collections import deque

dq = deque([0, 1])
dq.append(2)       # in fondo
dq.appendleft(-1)  # in testa
print(dq, dq[0], dq[-1])  # -> deque([-1, 0, 1, 2]) -1 2
dq.popleft()
dq.pop()
print(dq)  # -> deque([0, 1])

hourly_orders = [1, 3, 2, 5]
dq = deque()
for i, value in enumerate(hourly_orders):
    while dq and hourly_orders[dq[-1]] <= value:
        dq.pop()
    dq.append(i)
print(list(dq), hourly_orders[dq[0]])  # -> [3] 5

deque([-1, 0, 1, 2]) -1 2
deque([0, 1])
[3] 5


## Errori comuni

- Lunghezza `r - l` invece di `r - l + 1`.
- Usare un `if` invece del `while` quando restringi: la finestra può dover perdere più elementi.
- `list.pop(0)` al posto di `deque.popleft()`: `O(n)` invece di `O(1)`.
- Salvare la sottostringa a ogni passo (`s[l:r+1]` costa `O(n)`): salva solo gli indici del migliore e taglia alla fine.